# Preprocessing, fitur, dan split anti-leakage
Notebook tipis. Kode: `remediaide_ml.data.{eedi,map,splits,translate}`, `features.{history,labels,encoding}`, `scripts/build_features.py`. Tes leakage: `tests/test_features_leakage.py`.

In [1]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from remediaide_ml.paths import TABLES, FIGURES, PROCESSED
pd.set_option("display.precision", 4)
def js(name): return json.loads((TABLES / name).read_text())

## Kartu data (dihasilkan `data/eedi_card.py`, `data/map_card.py`)

In [2]:
display(Markdown((TABLES / 'data_card_eedi.md').read_text()))

# Data card — Eedi NeurIPS 2020, Task 3&4

Dihasilkan otomatis oleh `python -m remediaide_ml.data.eedi_card` dari `data/interim/eedi_t34.parquet`.
Lisensi: CC BY-NC-ND 4.0 (data olahan tidak dibagikan; hanya skrip).

## Ringkasan
- Baris mentah train_task_3_4: 1,382,727; duplikat AnswerId dibuang: 0; tidak konsisten IsCorrect dibuang: 0
- Baris bersih: 1,382,727; siswa: 4,918; soal: 948; KC (SubjectId terdalam): 57; baris tanpa KC: 0
- Baris tanpa answer metadata: 0
- Rentang tanggal: 2019-09-06 14:05:00 s.d. 2020-05-01 08:33:00
- Akurasi rata-rata (IsCorrect): 0.5373
- **Confidence kosong: 74.95% baris**; per siswa median 90.53%, siswa tanpa confidence sama sekali: 12.89%
- Panjang sekuens per siswa: median 239, p90 555, maks 827
- Percobaan ulang (attempt_no > 1): 0.00% baris

## Distribusi Confidence (baris yang terisi)

| Confidence | n | % | CRI (D-03) |
| ---: | ---: | ---: | ---: |
| 0 | 28,733 | 8.29% | 0 |
| 25 | 17,405 | 5.02% | 1 |
| 50 | 53,043 | 15.31% | 3 |
| 75 | 87,778 | 25.34% | 4 |
| 100 | 159,469 | 46.03% | 5 |

Nilai unik Confidence: 5

## CRI dan kuadran B0 (baris berkeyakinan)

- P (benar, yakin): 199,120 (57.48%)
- L (benar, tidak yakin): 16,361 (4.72%)
- M (salah, yakin): 101,170 (29.20%)
- T (salah, tidak yakin): 29,777 (8.60%)

## Atribut siswa (hanya untuk analisis fairness, bukan fitur)

| Gender | siswa | % |
| --- | ---: | ---: |
| 0 | 1,874 | 38.10% |
| 1 | 1,524 | 30.99% |
| 2 | 1,513 | 30.76% |
| 3 | 7 | 0.14% |

| PremiumPupil | siswa | % |
| --- | ---: | ---: |
| 0.0 | 1,512 | 30.74% |
| 1.0 | 601 | 12.22% |
| <NA> | 2,805 | 57.04% |

Kode Gender (panduan data Eedi): 0 tidak disebut, 1 perempuan, 2 laki-laki, 3 lainnya.


In [3]:
display(Markdown((TABLES / 'data_card_map.md').read_text()))

# Data card — MAP (Charting Student Math Misunderstandings)

Dihasilkan otomatis oleh `python -m remediaide_ml.data.map_card` dari `data/interim/map.parquet`.

## Ringkasan
- Baris: 36,696; soal: 15; target unik (Category:Misconception): 65
- Label miskonsepsi unik: 35 (pasangan soal×label: 39)
- Baris dengan konflik prefiks benar/salah vs jawaban (noise label, dipertahankan, ditandai `prefix_conflict`): 18 (0.049%)
- **True_Misconception (jebakan jawaban benar): 403 (1.10%)**
- Panjang alasan (kata): median 13, p90 26, maks 177

Catatan: paper dataset menyebut 27% alasan sintetis (parafrase Coedit-XL), tetapi file Kaggle tidak memberi penanda asli/sintetis, sehingga evaluasi terpisah tidak mungkin. Paper melaporkan 52.463 alasan; train.csv publik hanya memuat 36,696 baris (sisanya diduga test tersembunyi Kaggle).

## Per kategori

| Category | n | % |
| --- | ---: | ---: |
| True_Correct | 14,802 | 40.34% |
| False_Misconception | 9,457 | 25.77% |
| False_Neither | 6,542 | 17.83% |
| True_Neither | 5,265 | 14.35% |
| True_Misconception | 403 | 1.10% |
| False_Correct | 227 | 0.62% |

## Per soal

| QuestionId | n | % benar (MC) | % Misconception | % True_Misconception | label miskonsepsi |
| --- | ---: | ---: | ---: | ---: | --- |
| 31772 | 4,857 | 48.3% | 33.4% | 0.33% | Incomplete, WNB |
| 31774 | 3,115 | 61.6% | 20.4% | 0.64% | Mult, SwapDividend, FlipChange |
| 31777 | 2,809 | 64.0% | 13.9% | 0.00% | Wrong_Fraction, Incomplete, Irrelevant |
| 31778 | 3,640 | 48.4% | 39.1% | 3.46% | Additive, Irrelevant, WNB |
| 32829 | 2,156 | 79.6% | 12.6% | 0.05% | Not_variable, Adding_terms, Inverse_operation |
| 32833 | 3,105 | 51.6% | 36.2% | 0.35% | Duplication, Inversion, Wrong_Operation |
| 32835 | 2,332 | 71.8% | 17.1% | 0.81% | Whole_numbers_larger, Longer_is_bigger, Ignores_zeroes, Shorter_is_bigger |
| 33471 | 1,542 | 56.7% | 28.3% | 0.39% | Wrong_fraction, Incomplete |
| 33472 | 2,800 | 59.0% | 23.3% | 0.25% | Denominator-only_change, Adding_across, Incorrect_equivalent_fraction_addition |
| 33474 | 1,766 | 35.7% | 38.7% | 0.40% | Subtraction, Division |
| 76870 | 1,186 | 42.8% | 32.5% | 0.25% | Unknowable, Definition, Interior |
| 89443 | 3,054 | 45.6% | 28.0% | 5.37% | Positive, Tacking |
| 91695 | 2,610 | 59.3% | 25.5% | 0.73% | Wrong_term, Firstterm |
| 104665 | 673 | 73.8% | 17.7% | 0.59% | Multiplying_by_4, Base_rate |
| 109465 | 1,051 | 52.6% | 18.7% | 0.00% | Scale, Certainty |

## Contoh per kategori (baris pertama per kategori, row_id terkecil)

- **False_Correct** (Q31772, jawaban `\( \frac{3}{6} \)`, label `NA`): "Because 3 of 9 are not shaded as a simplified fraction it would be 1 3rd"
- **False_Misconception** (Q31772, jawaban `\( \frac{3}{6} \)`, label `WNB`): "3 IS SHADED AND 6 IS NOT"
- **False_Neither** (Q31772, jawaban `\( \frac{3}{6} \)`, label `NA`): "! simplefined it and that was what i got"
- **True_Correct** (Q31772, jawaban `\( \frac{1}{3} \)`, label `NA`): "0ne third is equal to tree nineth"
- **True_Misconception** (Q31772, jawaban `\( \frac{1}{3} \)`, label `Incomplete`): "3 out of 9 parts aren't shaded."
- **True_Neither** (Q31772, jawaban `\( \frac{1}{3} \)`, label `NA`): "1 3rd is half of 3 6th, so it is simplee to understand."


## Fitur Lapis 1 (hanya masa lalu siswa; encoding target out-of-fold)

In [4]:
info = json.loads((PROCESSED / 'eedi_features_info.json').read_text())
{k: v for k, v in info.items() if not isinstance(v, list)}

{'rows': 1382727,
 'sec_history': 47.6,
 'sec_labels': 0.5,
 'sec_total': 51.0,
 'rows_with_binary_label': 1139163,
 'rows_with_quadrant_label': 335216,
 'rows_with_quadrant_label_and_cur_conf': 276443,
 'rows_excluded_lt2_future': 243564,
 'fut_quadrant_dist': {'P': 0.5383, 'M': 0.2743, 'T': 0.1154, 'L': 0.072}}

In [5]:
from remediaide_ml.features.history import FEATURES_A, FEATURES_B_CUR, FEATURES_OOF, FEATURES_B_OOF
pd.DataFrame({'kelompok': ['riwayat (framing a)', 'jawaban saat ini (framing b)', 'encoding OOF', 'encoding OOF (b)'],
              'fitur': [', '.join(FEATURES_A), ', '.join(FEATURES_B_CUR), ', '.join(FEATURES_OOF), ', '.join(FEATURES_B_OOF)]})

,kelompok,fitur
0,riwayat (framing a),"n_prev, acc_prev, n_prev_kc, acc_prev_kc, ewm_..."
1,jawaban saat ini (framing b),"cur_correct, cur_cri, cur_has_conf, cur_quadra..."
2,encoding OOF,"item_diff_oof, kc_diff_oof"
3,encoding OOF (b),cur_option_share_oof


## Split: 20% siswa Eedi test terkunci, GroupKFold(5) per siswa; MAP StratifiedKFold + GroupKFold per soal (D-04, D-06)

In [6]:
sp = pd.read_parquet(PROCESSED / 'eedi_splits.parquet'); sp.fold.value_counts().sort_index().rename('siswa per fold (-1 = test)')

fold
-1    984
 0    787
 1    787
 2    787
 3    787
 4    786
Name: siswa per fold (-1 = test), dtype: int64

In [7]:
m = pd.read_parquet(PROCESSED / 'map_splits.parquet'); m.groupby('fold_question').QuestionId.unique()

fold_question
0    [31777, 31778, 109465]
1     [31772, 33471, 89443]
2     [32829, 32835, 33474]
3    [76870, 91695, 104665]
4     [31774, 32833, 33472]
Name: QuestionId, dtype: object

## Terjemahan Indonesia + QA manual 100 sampel (D-08)

In [8]:
display(Markdown((TABLES.parent / 'translation_card.md').read_text()))

# Kartu terjemahan — MAP Inggris → Indonesia (T32)

| Item | Nilai |
| --- | --- |
| Model | Helsinki-NLP/opus-mt-en-id (Apache-2.0), beam 4, batch 32, MPS (D-08) |
| Kode | `src/remediaide_ml/data/translate.py`, run `20261118-2129_translate_map` |
| Output | `data/processed/map_id.parquet` (36.696 baris; tidak di-commit) |
| Waktu | 35.668 alasan unik dalam 2.744 detik (~13 alasan/detik) |

## Prosedur
1. Teks dipecah per kalimat (Marian membuang kalimat akhir pada input panjang; ditemukan saat uji 20 contoh).
2. LaTeX (`\( … \)`, `\frac{a}{b}`, `$…$`) dan pecahan `a/b` diganti placeholder `ZQX<n>Z`, lalu dikembalikan.
   Angka yang menempel ke huruf (`2y`, `8th`) tidak dilindungi karena placeholder di dalam kata rusak.
3. Placeholder hilang → segmen diterjemahkan ulang tanpa proteksi (326 segmen).
4. Pengaman halusinasi: output > 2,5× panjang sumber + 20 karakter → ulang greedy + no_repeat_ngram=3; jika masih,
   teks sumber dipertahankan dan ditandai. opus-mt-en-id kadang menghasilkan teks keagamaan panjang yang tak terkait
   (ditemukan pada soal 31772).

## Diagnostik otomatis (seluruh data, per string unik)
| Kolom | Unik | Angka tidak cocok | Tidak berubah | Segmen diulang | Halusinasi (sumber dipertahankan) |
| --- | ---: | ---: | ---: | ---: | ---: |
| QuestionText | 15 | 0 | 1 | 1 | 1 |
| MC_Answer | 49 | 0 | 21 | 0 | 0 |
| StudentExplanation | 35.668 | 845 (2,4%) | 246 (0,7%) | 326 | 18 (0,05%) |

## QA manual — 100 sampel acak berstrata per kategori (seed 20261118)
Sumber: `reports/tables/translation_qa.csv` (EN, ID, tingkat, jenis kesalahan, catatan per sampel). Penilai: Claude Code.
Rubrik: makna penalaran terjaga, matematika utuh, bahasa wajar. **Berat** = guru atau model bisa salah membaca penalaran.

| Tingkat | n |
| --- | ---: |
| Baik | 46 |
| Kesalahan kecil | 34 |
| **Kesalahan berat** | **20** |

| Jenis kesalahan (bisa lebih dari satu per sampel) | n |
| --- | ---: |
| Potongan tidak diterjemahkan ("i think", "thats") | 16 |
| Terjemahan harfiah ("times table" → "meja waktu", "left" → "kiri", "letter" → "surat") | 15 |
| Istilah matematika keliru ("1 third" → "1 ketiga", "fractions" → "fraksi", "number" → "nomor") | 9 |
| Bahasa kaku | 9 |
| Angka/ekspresi rusak ("-8 - -5" → "-8 -5", "0.0001" → "0, 0001", "6.0001" → "6.001") | 7 |
| Bagian hilang | 4 |

- Cek digit otomatis vs penilaian manual: menangkap 5 dari 7 sampel dengan angka rusak, dengan 4 positif palsu
  (mis. "4" → "empat", "negative three" → "-3").
- Kesalahan berat per kategori: False_Correct 40% (6/15), kategori lain 12–18%.
- **Opsi jawaban teks soal peluang (109465) salah makna:** "Unlikely" → "Tidak mungkin" (= impossible),
  "Certain" → "Tentu", "Likely" → "Sepertinya". Tidak diperbaiki diam-diam; dilaporkan di sini dan di E4.

## Konsekuensi
- E4 mengukur ketahanan model terhadap **terjemahan mesin dengan ~20% kesalahan berat**, bukan tulisan asli siswa
  Indonesia. Validasi native menunggu pilot (E9).
- Untuk instrumen pilot (T70), soal dan opsi diterjemahkan ulang secara manual oleh tim, bukan memakai keluaran ini.


In [9]:
q = pd.read_csv(TABLES / 'translation_qa.csv'); q.iloc[:, :6].head(10)

,row_id,Category,en,id,cek_digit_otomatis,tingkat
0,3891,False_Correct,There are 9 triangles in total which is your d...,Ada 9 segitiga secara total yang penyebut Anda...,True,minor
1,11972,False_Correct,"To get to 10 to 15 you divide by 2, times by 3...",Untuk mendapatkan 10 sampai 15 Anda membagi de...,True,ok
2,19747,False_Correct,I think it is b because 6.2000 = that which is...,Saya pikir itu adalah b karena 6.2000 = yang l...,False,major
3,25264,False_Correct,because u do 1/3x2/5= 11/15.u do 5x1/3=5/15 th...,karena kau melakukan 1/3x2/5=11/15.u lakukan 5...,False,major
4,22997,False_Correct,15 because you divided 24 by 8 and times it by...,15 karena Anda dibagi 24 dengan 8 dan kali 3 m...,True,ok
5,25607,False_Correct,because you need to times three by five as the...,karena Anda perlu dikali tiga kali lima karena...,True,major
6,32915,False_Correct,i think this because every time a new pattern ...,i think ini karena setiap kali pola baru terja...,True,minor
7,22928,False_Correct,"1 / 8 of 24 wouldd be 3 so 3 /8 of 24, would n...",1 / 8 dari 24 akan menjadi 3 sehingga 3 /8 dar...,True,ok
8,5421,False_Correct,i think this because if you split a chocolate ...,i think ini karena jika Anda membagi cokelat b...,True,minor
9,11230,False_Correct,i divided it into fifths and timed it by three.,i dibagi menjadi lima dan waktu itu oleh tiga.,True,major
